# Runnables in LangChain

## 1. What is a Runnable?

A **Runnable** is one of the core abstractions in LangChain used to represent a unit of work that can be executed.

A Runnable can represent:

- A prompt template
- A Chat Model / LLM
- An output parser
- A retriever
- A Python function
- A complete chain
- A combination of multiple components

### Core Idea

> Anything that implements the Runnable interface can be executed and composed with other Runnables.

A typical LangChain pipeline looks like:

    Input
      ↓
    Prompt
      ↓
    LLM
      ↓
    Output Parser
      ↓
    Final Output

Each component can be a Runnable.

---

# 2. Why Runnables are Important

Runnables provide a **common interface** for many LangChain components.

Instead of learning completely different execution methods for:

- Prompts
- Models
- Parsers
- Retrievers
- Functions
- Chains

you can use a common Runnable interface.

### Benefits

Runnables make LangChain applications:

- Modular
- Composable
- Reusable
- Easier to debug
- Easier to stream
- Easier to execute asynchronously
- Easier to execute in parallel
- Easier to configure
- Easier to trace
- Easier to maintain

---

# 3. Runnable Mental Model

The simplest way to understand a Runnable is:

    Input
      ↓
    Runnable
      ↓
    Output

For example:

    User Question
         ↓
    Prompt Template
         ↓
    Chat Model
         ↓
    Output Parser
         ↓
    Answer

Each stage can be a Runnable.

---

# 4. Runnable Interface

The most important execution methods are:

| Method | Purpose |
|---|---|
| `invoke()` | Execute one input synchronously |
| `ainvoke()` | Execute one input asynchronously |
| `batch()` | Process multiple inputs |
| `abatch()` | Process multiple inputs asynchronously |
| `stream()` | Stream output synchronously |
| `astream()` | Stream output asynchronously |

These methods are the foundation of Runnable execution.

---

# 5. `invoke()`

`invoke()` executes a Runnable synchronously for a single input.

### Example

    from langchain_core.runnables import RunnableLambda

    runnable = RunnableLambda(lambda x: x * 2)

    result = runnable.invoke(5)

    print(result)

Output:

    10

### Flow

    5
    ↓
    RunnableLambda
    ↓
    10

### Use `invoke()` when:

- You have one input.
- You want synchronous execution.
- You want the final result directly.

---

# 6. `ainvoke()`

`ainvoke()` is the asynchronous version of `invoke()`.

### Example

    import asyncio
    from langchain_core.runnables import RunnableLambda

    runnable = RunnableLambda(lambda x: x * 2)

    async def main():
        result = await runnable.ainvoke(5)
        print(result)

    asyncio.run(main())

Output:

    10

### Use `ainvoke()` when:

- Building asynchronous applications
- Working with `asyncio`
- Building FastAPI applications
- Handling concurrent operations
- Building scalable AI applications

---

# 7. `batch()`

`batch()` allows a Runnable to process multiple inputs.

### Example

    from langchain_core.runnables import RunnableLambda

    runnable = RunnableLambda(lambda x: x * 2)

    results = runnable.batch([1, 2, 3, 4])

    print(results)

Output:

    [2, 4, 6, 8]

Instead of:

    runnable.invoke(1)
    runnable.invoke(2)
    runnable.invoke(3)
    runnable.invoke(4)

you can use:

    runnable.batch([1, 2, 3, 4])

---

# 8. `abatch()`

`abatch()` is the asynchronous version of `batch()`.

### Example

    results = await runnable.abatch([1, 2, 3, 4])

It is useful when multiple inputs need to be processed asynchronously.

### Mental model

    Multiple Inputs
          ↓
       abatch()
          ↓
    Async Processing
          ↓
    Multiple Outputs

---

# 9. `stream()`

`stream()` allows a Runnable to return results incrementally instead of waiting for the entire result.

This is especially important for LLM applications.

### Without streaming

    User
     ↓
    LLM
     ↓
    Wait...
     ↓
    Complete Answer

### With streaming

    User
     ↓
    LLM
     ↓
    Chunk 1
     ↓
    Chunk 2
     ↓
    Chunk 3
     ↓
    Chunk 4
     ↓
    Complete Answer

### Example

    for chunk in runnable.stream(input):
        print(chunk)

Streaming is useful for:

- Chat applications
- Interactive AI interfaces
- Real-time responses
- Long LLM responses

---

# 10. `astream()`

`astream()` is the asynchronous version of `stream()`.

### Example

    async for chunk in runnable.astream(input):
        print(chunk)

It is useful for:

- Async chat applications
- Streaming APIs
- Web applications
- Real-time AI systems

---

# 11. Easy Memory Trick

Remember the Runnable execution methods like this:

    invoke   → one input
    batch    → many inputs
    stream   → chunks

And:

    a + method = asynchronous version

Therefore:

    invoke()   → one, synchronous
    ainvoke()  → one, asynchronous

    batch()    → many, synchronous
    abatch()   → many, asynchronous

    stream()   → streaming, synchronous
    astream()  → streaming, asynchronous

---

# 12. RunnableLambda

`RunnableLambda` converts a normal Python function into a Runnable.

### Normal Python function

    def add_one(x):
        return x + 1

Convert it into a Runnable:

    from langchain_core.runnables import RunnableLambda

    runnable = RunnableLambda(add_one)

Now it can be executed using:

    result = runnable.invoke(5)

    print(result)

Output:

    6

### Why is RunnableLambda useful?

It allows custom Python logic to become part of a LangChain pipeline.

For example:

    Input
      ↓
    Python Function
      ↓
    Prompt
      ↓
    LLM
      ↓
    Parser
      ↓
    Output

This is very useful when you need custom preprocessing or postprocessing.

---

# 13. RunnablePassthrough

`RunnablePassthrough` returns the input unchanged.

### Example

    from langchain_core.runnables import RunnablePassthrough

    runnable = RunnablePassthrough()

    result = runnable.invoke("Hello")

    print(result)

Output:

    Hello

### Main purpose

`RunnablePassthrough` is especially useful when creating parallel Runnable structures.

For example:

    {
        "question": RunnablePassthrough(),
        "context": retriever
    }

The original input is:

1. Passed directly to `"question"`
2. Sent to the retriever to generate `"context"`

---

# 14. RunnableSequence

`RunnableSequence` executes Runnables sequentially.

Conceptually:

    Runnable A
        ↓
    Runnable B
        ↓
    Runnable C
        ↓
    Output

For example:

    chain = prompt | model | parser

This means:

    Prompt
      ↓
    Model
      ↓
    Parser

The output of one Runnable becomes the input of the next Runnable.

---

# 15. LCEL

**LCEL** stands for:

> LangChain Expression Language

LCEL provides a simple syntax for composing Runnables.

The most common syntax is:

    chain = prompt | model | parser

The `|` operator represents composition.

It means:

    Output of A → Input of B

Therefore:

    prompt | model | parser

means:

    Prompt
      ↓
    Model
      ↓
    Parser

---

# 16. Why LCEL is Useful

Without Runnable composition, you might write:

    prompt_result = prompt.invoke(input)

    model_result = model.invoke(prompt_result)

    final_result = parser.invoke(model_result)

With LCEL:

    chain = prompt | model | parser

    final_result = chain.invoke(input)

LCEL makes chains:

- Shorter
- Easier to understand
- Easier to compose
- Easier to reuse
- Easier to stream
- Easier to execute asynchronously

---

# 17. Important Concept: A Chain is Also a Runnable

Consider:

    chain = prompt | model | parser

The resulting `chain` is itself a Runnable.

Therefore, you can use:

    chain.invoke(input)

You can also use:

    chain.batch(inputs)

    chain.stream(input)

    await chain.ainvoke(input)

This is one of the most important concepts in LangChain.

### Mental Model

    Small Runnable
          ↓
    Small Runnable
          ↓
        Chain
          ↓
    Larger Runnable
          ↓
      Application

A Runnable can therefore contain other Runnables.

---

# 18. RunnableParallel

`RunnableParallel` executes multiple Runnables using the same input.

### Example

    from langchain_core.runnables import RunnableParallel, RunnableLambda

    parallel = RunnableParallel(
        doubled=RunnableLambda(lambda x: x * 2),
        squared=RunnableLambda(lambda x: x ** 2)
    )

    result = parallel.invoke(5)

    print(result)

Output:

    {
        "doubled": 10,
        "squared": 25
    }

### Architecture

    Input = 5
          │
          ├──→ doubled → 10
          │
          └──→ squared → 25

Both Runnable branches receive the same input.

---

# 19. RunnableParallel vs RunnableSequence

These two concepts are very important.

### RunnableSequence

Execution happens one after another.

    Input
      ↓
      A
      ↓
      B
      ↓
      C
      ↓
    Output

### RunnableParallel

Multiple branches operate on the same input.

             ┌──→ A
    Input ────┤
             └──→ B

### Simple difference

    Sequence  = A → B → C

    Parallel  = A and B at the same time

---

# 20. Dictionary Syntax for Parallel Runnables

LangChain provides a convenient shorthand for parallel execution.

Example:

    chain = {
        "original": RunnablePassthrough(),
        "upper": RunnableLambda(lambda x: x.upper())
    }

Then:

    result = chain.invoke("hello")

Result:

    {
        "original": "hello",
        "upper": "HELLO"
    }

Conceptually:

    Input
      │
      ├──→ original → "hello"
      │
      └──→ upper → "HELLO"

---

# 21. RunnableParallel in RAG

`RunnableParallel` is extremely important in Retrieval-Augmented Generation.

Suppose the user asks:

    What is LangChain?

A RAG system needs:

1. The original question
2. Relevant documents from the retriever

We can construct:

    {
        "context": retriever,
        "question": RunnablePassthrough()
    }

### Architecture

                     ┌──→ Retriever
                     │       ↓
                     │    Context
    Question ────────┤
                     │
                     └──→ Passthrough
                             ↓
                          Question

Then:

    Context + Question
           ↓
         Prompt
           ↓
          LLM
           ↓
      Output Parser
           ↓
         Answer

---

# 22. RunnableBranch

`RunnableBranch` is used for conditional execution.

It allows a Runnable to select a path based on the input.

### Concept

                     ┌──→ Condition 1 → Runnable A
                     │
    Input → Branch ──┼──→ Condition 2 → Runnable B
                     │
                     └──→ Default → Runnable C

### Example

    from langchain_core.runnables import RunnableBranch, RunnableLambda

    branch = RunnableBranch(
        (
            lambda x: x > 10,
            RunnableLambda(lambda x: "Greater than 10")
        ),
        (
            lambda x: x <= 10,
            RunnableLambda(lambda x: "10 or less")
        )
    )

    result = branch.invoke(15)

    print(result)

Output:

    Greater than 10

---

# 23. RunnableBranch in AI Applications

RunnableBranch can be used for routing.

For example:

    User Query
         ↓
    Classifier
         ↓
    ┌────┼────────────┐
    ↓    ↓            ↓
    Technical       General       Support
       ↓               ↓             ↓
    Tech Chain     General Chain  Support Chain

This is called:

- Routing
- Conditional execution
- Query routing

---

# 24. RunnableAssign

`RunnableAssign` is used to add new values to an existing dictionary.

Conceptually:

    Existing Dictionary
           ↓
    RunnableAssign
           ↓
    Updated Dictionary

For example, an input might be:

    {
        "question": "What is LangChain?"
    }

A RunnableAssign operation can calculate another value and add it:

    {
        "question": "What is LangChain?",
        "category": "AI Framework"
    }

This is useful when progressively building the input needed by later stages.

---

# 25. RunnablePick

`RunnablePick` is used to select specific values from a dictionary.

Suppose we have:

    {
        "question": "What is LangChain?",
        "context": "LangChain is a framework...",
        "metadata": {
            "source": "documentation"
        }
    }

If a later Runnable only needs:

    question

we can use a pick operation to select the required value.

Conceptually:

    Dictionary
        ↓
    RunnablePick
        ↓
    Selected Value

This helps control what data flows into the next Runnable.

---

# 26. RunnableMap

`RunnableMap` is associated with mapping an input across multiple Runnable operations.

Modern LangChain code commonly uses dictionary-based Runnable composition for this type of parallel mapping.

Example:

    chain = {
        "question": RunnablePassthrough(),
        "processed": RunnableLambda(lambda x: x.lower())
    }

The same input can therefore be transformed into multiple outputs.

---

# 27. RunnableConfig

Runnables can accept runtime configuration through `RunnableConfig`.

Configuration can be useful for:

- Tags
- Metadata
- Run names
- Concurrency
- Tracing
- Runtime configuration

Example:

    config = {
        "tags": ["demo"],
        "metadata": {
            "user_type": "student"
        }
    }

    result = chain.invoke(
        input,
        config=config
    )

---

# 28. Tags

Tags help categorize Runnable executions.

Example:

    config = {
        "tags": ["production", "rag"]
    }

Tags can help when inspecting execution traces and debugging applications.

---

# 29. Metadata

Metadata allows additional information to be attached to a Runnable execution.

Example:

    config = {
        "metadata": {
            "request_type": "question_answering"
        }
    }

Metadata can be useful for:

- Debugging
- Tracing
- Analytics
- Observability

Avoid putting unnecessary sensitive information into metadata.

---

# 30. Run Names

A Runnable execution can be given a meaningful name.

Example:

    chain = chain.with_config(
        run_name="question_answering_chain"
    )

Meaningful names make complex execution traces easier to understand.

---

# 31. Runnable Binding

Runnables can be configured with fixed parameters using `.bind()`.

Conceptually:

    configured_runnable = runnable.bind(...)

The resulting Runnable can then be reused with those parameters already configured.

This is useful when a particular model or Runnable should always be called with certain settings.

The exact parameters depend on the Runnable or model integration.

---

# 32. Runnable Retry

Runnables can be configured with retry behavior.

Conceptually:

    runnable.with_retry(...)

This is useful for operations that can temporarily fail, such as:

- API calls
- Network requests
- Model requests
- External services

### Architecture

    Input
      ↓
    Runnable
      ↓
    Success?
     /   \
   Yes    No
    ↓      ↓
  Output  Retry
           ↓
        Runnable

Retries should be configured carefully because excessive retries can:

- Increase latency
- Increase API costs
- Delay failure handling

---

# 33. Runnable Fallbacks

A Runnable can also have fallback behavior.

Conceptually:

    Primary Runnable
          ↓
       Failure
          ↓
    Fallback Runnable
          ↓
        Output

This is useful when you have:

- A primary model
- A backup model
- A primary API
- A backup API
- A primary processing method
- A backup processing method

Conceptually:

    primary.with_fallbacks([fallback])

---

# 34. Complete LLM Runnable Chain

A basic LangChain application can be constructed like this:

    from langchain_core.prompts import ChatPromptTemplate
    from langchain_openai import ChatOpenAI
    from langchain_core.output_parsers import StrOutputParser

    prompt = ChatPromptTemplate.from_template(
        "Explain {topic} in simple words."
    )

    model = ChatOpenAI()

    parser = StrOutputParser()

    chain = prompt | model | parser

    result = chain.invoke({
        "topic": "Runnables in LangChain"
    })

    print(result)

### Architecture

    Input Dictionary
          ↓
    ChatPromptTemplate
          ↓
       ChatOpenAI
          ↓
    StrOutputParser
          ↓
        String

---

# 35. Runnable with Custom Python Logic

We can insert custom Python processing into a Runnable chain.

Example:

    from langchain_core.runnables import RunnableLambda

    preprocess = RunnableLambda(
        lambda x: x.strip().lower()
    )

    chain = preprocess | prompt | model | parser

Architecture:

    User Input
        ↓
    Preprocessing
        ↓
    Prompt
        ↓
    Model
        ↓
    Parser
        ↓
    Final Answer

This is useful for:

- Cleaning input
- Formatting data
- Validating input
- Transforming output
- Applying business logic

---

# 36. Combining Sequential and Parallel Runnables

Runnables can be combined into complex architectures.

Example:

                     ┌──→ Retriever ──→ Context
                     │
    Question ────────┼──→ Passthrough ─→ Question
                     │
                     └──→ Classifier ──→ Category
                                      ↓
                              Combined Dictionary
                                      ↓
                                    Prompt
                                      ↓
                                     LLM
                                      ↓
                                  Parser
                                      ↓
                                    Answer

This demonstrates that LangChain pipelines are not restricted to simple linear chains.

---

# 37. Runnables in RAG

A modern RAG pipeline can be represented using Runnables.

    User Question
          ↓
    ┌─────┴──────────┐
    ↓                ↓
    Retriever    Original Question
    ↓                ↓
    Documents     Passthrough
    └─────┬──────────┘
          ↓
        Prompt
          ↓
         LLM
          ↓
    Output Parser
          ↓
        Answer

A conceptual implementation:

    retrieval_chain = (
        {
            "context": retriever,
            "question": RunnablePassthrough()
        }
        | prompt
        | model
        | StrOutputParser()
    )

Then:

    answer = retrieval_chain.invoke(
        "What is LangChain?"
    )

---

# 38. Runnables and Streaming

One major advantage of Runnable-based chains is that streaming can be applied to the chain.

For example:

    for chunk in chain.stream(input):
        print(chunk)

Instead of waiting for the complete answer, the application can receive chunks progressively.

This is particularly useful for:

- Chatbots
- AI assistants
- Interactive interfaces
- Long responses

---

# 39. Runnables and Async Execution

Runnable chains can also be executed asynchronously.

Example:

    result = await chain.ainvoke(input)

This is important for modern backend applications where many operations may happen concurrently.

For example:

    User Request
         ↓
    Async Runnable
         ↓
    External API
         ↓
    Model
         ↓
    Response

---

# 40. Runnables and Batch Processing

Suppose we have many questions:

    questions = [
        "What is LangChain?",
        "What is LCEL?",
        "What is RAG?",
        "What is a Runnable?"
    ]

Instead of processing each separately:

    chain.invoke(questions[0])
    chain.invoke(questions[1])
    chain.invoke(questions[2])
    chain.invoke(questions[3])

we can use:

    results = chain.batch(questions)

This is useful for:

- Dataset processing
- Evaluation
- Bulk summarization
- Batch classification
- Large-scale AI workflows

---

# 41. Runnable Composition Operators

The most important composition operator is:

    |

It represents sequential composition.

Example:

    prompt | model | parser

Meaning:

    prompt
      ↓
    model
      ↓
    parser

You can combine this with dictionaries and other Runnable structures to create more complex workflows.

---

# 42. Runnable Architecture

A useful way to visualize Runnable architecture is:

    ┌──────────────────────────────────────┐
    │              Runnable                │
    │                                      │
    │  Input → Processing → Output         │
    │                                      │
    │  invoke()                            │
    │  ainvoke()                           │
    │  batch()                             │
    │  abatch()                            │
    │  stream()                            │
    │  astream()                           │
    └──────────────────────────────────────┘

A complex chain is simply a composition of these units.

---

# 43. Runnable Types - Quick Revision

| Runnable | Main Purpose |
|---|---|
| `RunnableLambda` | Convert Python function into Runnable |
| `RunnablePassthrough` | Pass input unchanged |
| `RunnableSequence` | Execute Runnables sequentially |
| `RunnableParallel` | Execute multiple branches using same input |
| `RunnableBranch` | Conditional routing |
| `RunnableAssign` | Add values to dictionary |
| `RunnablePick` | Select values from dictionary |
| `RunnableMap` | Map input across Runnable operations |

---

# 44. Runnable Execution Methods - Quick Revision

| Method | Meaning |
|---|---|
| `invoke()` | One synchronous execution |
| `ainvoke()` | One asynchronous execution |
| `batch()` | Multiple synchronous executions |
| `abatch()` | Multiple asynchronous executions |
| `stream()` | Synchronous streaming |
| `astream()` | Asynchronous streaming |

### Memory Trick

    One       → invoke
    Many      → batch
    Streaming → stream
    Async     → add "a"

---

# 45. Sequence vs Parallel vs Branch

| Concept | Purpose |
|---|---|
| Sequence | Run A → B → C |
| Parallel | Run multiple paths from same input |
| Branch | Choose one path based on condition |

### Sequence

    Input → A → B → C → Output

### Parallel

             ┌→ A → Output A
    Input ───┤
             └→ B → Output B

### Branch

                   ┌→ A
    Input → Condition ├→ B
                   └→ Default

---

# 46. Runnable vs Function

A normal Python function:

    def double(x):
        return x * 2

A Runnable:

    runnable = RunnableLambda(double)

Now the function can participate in LangChain's Runnable ecosystem.

For example:

    runnable.invoke(5)

    runnable.batch([1, 2, 3])

    runnable.stream(...)

Depending on the Runnable implementation, it can also participate in async execution and composition.

---

# 47. Runnable vs Chain

A useful way to think about the relationship is:

    Runnable
       ↓
    Basic building block

    Chain
       ↓
    Combination of Runnables

However, because a composed chain is itself a Runnable, the relationship is recursive:

    Runnable
       ↓
    Runnable
       ↓
    Runnable
       ↓
    Chain
       ↓
    Runnable
       ↓
    Larger Application

---

# 48. Why Runnables Matter in Modern LangChain

Runnables are important because they provide the foundation for composing LangChain applications.

They allow developers to combine:

    Prompt
      +
    Model
      +
    Retriever
      +
    Python Logic
      +
    Parser
      +
    Conditional Logic
      +
    Parallel Processing

into a single executable workflow.

---

# 49. Real-World Example

Imagine building an AI customer-support system.

The workflow could be:

    User Message
         ↓
    Input Preprocessor
         ↓
    Intent Classifier
         ↓
    ┌──────────────┬───────────────┬───────────────┐
    ↓              ↓               ↓
    Billing      Technical       General
    ↓              ↓               ↓
    Billing      Technical       General
    Chain        Chain            Chain
    └──────────────┴───────────────┴───────────────┘
                         ↓
                    Final Response

This architecture can be built using Runnables.

---

# 50. Practical Example: Simple Runnable Pipeline

    from langchain_core.runnables import RunnableLambda

    clean = RunnableLambda(
        lambda x: x.strip()
    )

    uppercase = RunnableLambda(
        lambda x: x.upper()
    )

    chain = clean | uppercase

    result = chain.invoke("  hello langchain  ")

    print(result)

Output:

    HELLO LANGCHAIN

### Execution

    "  hello langchain  "
              ↓
            clean
              ↓
      "hello langchain"
              ↓
          uppercase
              ↓
      "HELLO LANGCHAIN"

This demonstrates the basic power of Runnable composition.

---

# 51. Practical Example: Parallel Processing

    from langchain_core.runnables import RunnableParallel, RunnableLambda

    analysis = RunnableParallel(
        uppercase=RunnableLambda(lambda x: x.upper()),
        lowercase=RunnableLambda(lambda x: x.lower()),
        length=RunnableLambda(lambda x: len(x))
    )

    result = analysis.invoke("LangChain")

Result:

    {
        "uppercase": "LANGCHAIN",
        "lowercase": "langchain",
        "length": 9
    }

Architecture:

                 ┌→ Uppercase
                 │
    "LangChain" ──┼→ Lowercase
                 │
                 └→ Length

---

# 52. Practical Example: RunnableBranch

    from langchain_core.runnables import RunnableBranch, RunnableLambda

    branch = RunnableBranch(
        (
            lambda x: len(x) > 10,
            RunnableLambda(lambda x: "Long input")
        ),
        RunnableLambda(lambda x: "Short input")
    )

    result = branch.invoke("Hello")

Output:

    Short input

Architecture:

    Input
      ↓
    Check Length
      ↓
    ┌──────────────┐
    │              │
    > 10           <= 10
    │              │
    ↓              ↓
    Long Input     Short Input

---

# 53. Common Mistakes

## Mistake 1: Confusing `invoke()` and `batch()`

Wrong mental model:

    batch() = one input

Correct:

    invoke() = one input
    batch()  = multiple inputs

---

## Mistake 2: Confusing Sequence and Parallel

Sequence:

    A → B → C

Parallel:

    A
    B
    C

executed from the same input.

---

## Mistake 3: Thinking Runnable is only an LLM

A Runnable is not limited to an LLM.

A Runnable can be:

- Prompt
- Model
- Parser
- Retriever
- Function
- Chain
- Branch
- Parallel workflow

---

## Mistake 4: Thinking a Chain and Runnable are completely separate concepts

A composed chain is itself a Runnable.

This means:

    chain.invoke()

works because the chain follows the Runnable interface.

---

## Mistake 5: Ignoring `RunnablePassthrough`

`RunnablePassthrough` is particularly important in RAG and parallel pipelines.

It allows the original input to continue through a branch unchanged.

---

# 54. Key Concepts to Remember

### Runnable

A standardized executable component.

### RunnableLambda

Turns a Python function into a Runnable.

### RunnablePassthrough

Returns the input unchanged.

### RunnableSequence

Runs Runnables sequentially.

### RunnableParallel

Runs multiple branches using the same input.

### RunnableBranch

Selects a path based on a condition.

### RunnableAssign

Adds values to an existing dictionary.

### RunnablePick

Selects specific values from a dictionary.

### LCEL

LangChain Expression Language used to compose Runnables.

### `|`

Represents sequential composition.

---

# 55. Most Important Interview Questions

## Q1. What is a Runnable in LangChain?

A Runnable is a standardized abstraction representing an executable unit of work in LangChain. Prompts, models, parsers, retrievers, functions, and composed chains can all participate in the Runnable interface.

---

## Q2. What is LCEL?

LCEL stands for **LangChain Expression Language**. It provides a declarative syntax for composing Runnables, commonly using the `|` operator.

Example:

    chain = prompt | model | parser

---

## Q3. What does the `|` operator mean in LangChain?

It represents sequential Runnable composition.

    A | B

means:

    Input → A → B → Output

---

## Q4. What is `RunnableLambda`?

`RunnableLambda` converts a Python callable into a Runnable.

Example:

    runnable = RunnableLambda(lambda x: x * 2)

---

## Q5. What is `RunnablePassthrough`?

It returns the input unchanged.

It is especially useful when the original input needs to be preserved while other Runnables process the same input.

---

## Q6. What is `RunnableParallel`?

It allows multiple Runnables to process the same input and produce multiple outputs.

---

## Q7. What is `RunnableBranch`?

It enables conditional routing between different Runnables.

---

## Q8. What is the difference between `invoke()` and `batch()`?

`invoke()` processes a single input.

`batch()` processes multiple inputs.

---

## Q9. What is the difference between `invoke()` and `ainvoke()`?

`invoke()` is synchronous.

`ainvoke()` is asynchronous.

---

## Q10. What is the difference between `stream()` and `invoke()`?

`invoke()` waits for the final result.

`stream()` can return output incrementally as it becomes available.

---

# 56. Final Mental Model

Remember Runnables using this structure:

    Runnables
       │
       ├── Execution
       │     ├── invoke()
       │     ├── ainvoke()
       │     ├── batch()
       │     ├── abatch()
       │     ├── stream()
       │     └── astream()
       │
       ├── Basic Runnables
       │     ├── RunnableLambda
       │     └── RunnablePassthrough
       │
       ├── Composition
       │     ├── RunnableSequence
       │     ├── RunnableParallel
       │     └── RunnableBranch
       │
       ├── Data Manipulation
       │     ├── RunnableAssign
       │     ├── RunnablePick
       │     └── RunnableMap
       │
       └── Configuration
             ├── RunnableConfig
             ├── Tags
             ├── Metadata
             ├── Retry
             ├── Fallbacks
             └── Binding

---

# 57. One-Line Summary

> **Runnables are LangChain's standardized building blocks for executing and composing prompts, models, retrievers, functions, parsers, and complete workflows using a common interface.**

The most important pattern to remember is:

    Input
      ↓
    Runnable
      ↓
    Runnable
      ↓
    Runnable
      ↓
    Output

And with LCEL:

    chain = prompt | model | parser

This simple concept forms the foundation for building modern LangChain pipelines, RAG systems, agents, routing systems, and complex AI workflows.